In [1]:
# Importing Libraries
import pandas as pd
import zipfile
import os

DATA_DIR = "../data/"
CLEAN_DIR = "../clean_data/"

os.makedirs(CLEAN_DIR, exist_ok=True)

print("Libraries loaded")
print(f"Clean data folder ready: {CLEAN_DIR}")

Libraries loaded
Clean data folder ready: ../clean_data/


In [2]:
#  Loading raw patents
with zipfile.ZipFile(DATA_DIR + "g_patent.tsv.zip") as z:
    inner = z.namelist()[0]
    df_patent = pd.read_csv(z.open(inner), sep='\t', nrows=100000, low_memory=False)

print(f"Patents loaded: {df_patent.shape}")
print(df_patent.dtypes)

Patents loaded: (100000, 8)
patent_id        int64
patent_type     object
patent_date     object
patent_title    object
wipo_kind       object
num_claims       int64
withdrawn        int64
filename        object
dtype: object


In [3]:
#Loading raw abstracts
with zipfile.ZipFile(DATA_DIR + "g_patent_abstract.tsv.zip") as z:
    inner = z.namelist()[0]
    df_abstract = pd.read_csv(z.open(inner), sep='\t', nrows=100000, low_memory=False)

print(f"Abstracts loaded: {df_abstract.shape}")
print(df_abstract.dtypes)

Abstracts loaded: (100000, 2)
patent_id           int64
patent_abstract    object
dtype: object


In [4]:
# Loading raw inventors
with zipfile.ZipFile(DATA_DIR + "g_inventor_disambiguated.tsv.zip") as z:
    inner = z.namelist()[0]
    df_inventor = pd.read_csv(z.open(inner), sep='\t', nrows=100000, low_memory=False)

print(f"Inventors loaded: {df_inventor.shape}")
print(df_inventor.dtypes)

Inventors loaded: (100000, 7)
patent_id                       object
inventor_sequence                int64
inventor_id                     object
disambig_inventor_name_first    object
disambig_inventor_name_last     object
gender_code                     object
location_id                     object
dtype: object


In [5]:
# Loading raw assignees
with zipfile.ZipFile(DATA_DIR + "g_assignee_disambiguated.tsv.zip") as z:
    inner = z.namelist()[0]
    df_assignee = pd.read_csv(z.open(inner), sep='\t', nrows=100000, low_memory=False)

print(f"Assignees loaded: {df_assignee.shape}")
print(df_assignee.dtypes)

Assignees loaded: (100000, 8)
patent_id                                   object
assignee_sequence                            int64
assignee_id                                 object
disambig_assignee_individual_name_first     object
disambig_assignee_individual_name_last      object
disambig_assignee_organization              object
assignee_type                              float64
location_id                                 object
dtype: object


In [6]:
# Loading full location file
with zipfile.ZipFile(DATA_DIR + "g_location_disambiguated.tsv.zip") as z:
    inner = z.namelist()[0]
    df_location = pd.read_csv(z.open(inner), sep='\t', low_memory=False)

print(f"Locations loaded: {df_location.shape}")
print(df_location.dtypes)

Locations loaded: (100452, 9)
location_id          object
disambig_city        object
disambig_state       object
disambig_country     object
latitude            float64
longitude           float64
county               object
state_fips          float64
county_fips         float64
dtype: object


In [7]:
# Clean Patents Table
df_patents_clean = df_patent[['patent_id','patent_type','patent_date','patent_title']].copy()

# merging with abstracts
df_patents_clean = df_patents_clean.merge(
    df_abstract[['patent_id','patent_abstract']],
    on='patent_id',
    how='left'
)

# rename columns
df_patents_clean.rename(columns={
    'patent_date':  'filing_date',
    'patent_title': 'title',
    'patent_abstract': 'abstract'
}, inplace=True)

# extracting year from filing_date
df_patents_clean['filing_date'] = pd.to_datetime(df_patents_clean['filing_date'], errors='coerce')
df_patents_clean['year'] = df_patents_clean['filing_date'].dt.year

# dropping rows with no patent_id or title
df_patents_clean.dropna(subset=['patent_id','title'], inplace=True)

# dropping duplicates
df_patents_clean.drop_duplicates(subset='patent_id', inplace=True)

print(f"Clean patents shape: {df_patents_clean.shape}")
display(df_patents_clean.head(3))

Clean patents shape: (100000, 6)


,patent_id,patent_type,filing_date,title,abstract,year
0,10000000,utility,2018-06-19,Coherent LADAR using intra-pixel quadrature de...,A frequency modulated (coherent) laser detecti...,2018
1,10000001,utility,2018-06-19,Injection molding machine and mold thickness c...,The injection molding machine includes a fixed...,2018
2,10000002,utility,2018-06-19,Method for manufacturing polymer film and co-e...,The present invention relates to: a method for...,2018


In [8]:
# Clean Inventors table
df_inventors_clean = df_inventor[['patent_id','inventor_id','disambig_inventor_name_first','disambig_inventor_name_last','location_id']].copy()

# combining first and last name
df_inventors_clean['full_name'] = (
    df_inventors_clean['disambig_inventor_name_first'].fillna('') + ' ' +
    df_inventors_clean['disambig_inventor_name_last'].fillna('')
).str.strip()

# joining with location to get country
df_inventors_clean = df_inventors_clean.merge(
    df_location[['location_id','disambig_country']],
    on='location_id',
    how='left'
)

# renaming columns
df_inventors_clean.rename(columns={
    'disambig_country': 'country'
}, inplace=True)

# keeping only needed columns
df_inventors_clean = df_inventors_clean[['inventor_id','patent_id','full_name','country']]

# dropping rows with no inventor_id
df_inventors_clean.dropna(subset=['inventor_id'], inplace=True)

# dropping duplicates
df_inventors_clean.drop_duplicates(inplace=True)

print(f"Clean inventors shape: {df_inventors_clean.shape}")
display(df_inventors_clean.head(3))

Clean inventors shape: (100000, 4)


,inventor_id,patent_id,full_name,country
0,fl:we_ln:jiang-165,D1006496,Wenjing Jiang,CN
1,fl:ei_ln:baumker-1,12029253,Eiko BÄUMKER,DE
2,fl:ri_ln:kroeger-1,6584128,Richard Kroeger,NaN


In [9]:
# Clean Companies Table
df_companies_clean = df_assignee[['assignee_id','disambig_assignee_organization','assignee_type','patent_id']].copy()

# renaming columns
df_companies_clean.rename(columns={
    'assignee_id': 'company_id',
    'disambig_assignee_organization': 'name'
}, inplace=True)

# dropping rows with no company name
df_companies_clean.dropna(subset=['name'], inplace=True)

# dropping duplicates
df_companies_clean.drop_duplicates(inplace=True)

print(f"Clean companies shape: {df_companies_clean.shape}")
display(df_companies_clean.head(3))

Clean companies shape: (98987, 4)


,company_id,name,assignee_type,patent_id
0,a9d256f6-26f6-4553-a8ae-1a1bf4de1030,Metal Works Ramat David,3.0,4488683
1,3ffdc6b4-9949-4395-9aae-f90cb3a18637,"DIVERGENT TECHNOLOGIES, INC.",2.0,11872626
2,bdcc4e9e-662e-4d11-bc1a-7beddcb4e431,U.S. Philips Corporation,2.0,5856666


In [10]:
# Building Relationships Table
# get patent_id and inventor_id
df_rel_inv = df_inventor[['patent_id','inventor_id']].copy()
df_rel_inv.dropna(inplace=True)
df_rel_inv.drop_duplicates(inplace=True)

# getting patent_id and assignee_id
df_rel_ass = df_assignee[['patent_id','assignee_id']].copy()
df_rel_ass.rename(columns={'assignee_id':'company_id'}, inplace=True)
df_rel_ass.dropna(inplace=True)
df_rel_ass.drop_duplicates(inplace=True)

# merging both on patent_id
df_relationships = df_rel_inv.merge(df_rel_ass, on='patent_id', how='inner')
df_relationships.drop_duplicates(inplace=True)

print(f"Relationships shape: {df_relationships.shape}")
print("")
display(df_relationships.head(3))

Relationships shape: (1151, 3)



,patent_id,inventor_id,company_id
0,11885288,fl:du_ln:moon-7,ea96c059-5267-4b52-ac70-4cd0915c519e
1,5444846,fl:na_ln:nagashima-3,dfdf1f1a-8f22-41dc-a7f0-855bde20ca67
2,6404762,fl:da_ln:meyer-50,f0236bcc-a68f-44e1-92d1-2d6b157ca97e


In [ ]:
# Saving all clean files
df_patents_clean.to_csv(CLEAN_DIR + "clean_patents.csv", index=False)
df_inventors_clean.to_csv(CLEAN_DIR + "clean_inventors.csv", index=False)
df_companies_clean.to_csv(CLEAN_DIR + "clean_companies.csv", index=False)
df_relationships.to_csv(CLEAN_DIR + "clean_relationships.csv", index=False)

print("All clean files saved")
print(f"clean_patents.csv       → {len(df_patents_clean):,} rows")
print(f"clean_inventors.csv     → {len(df_inventors_clean):,} rows")
print(f"clean_companies.csv     → {len(df_companies_clean):,} rows")
print(f"clean_relationships.csv → {len(df_relationships):,} rows")

All clean files saved
clean_patents.csv       → 100,000 rows
clean_inventors.csv     → 100,000 rows
clean_companies.csv     → 98,987 rows
clean_relationships.csv → 1,151 rows


In [12]:
# Verifying that clean files exist
for f in ["clean_patents.csv","clean_inventors.csv","clean_companies.csv","clean_relationships.csv"]:
    path = CLEAN_DIR + f
    exists = os.path.exists(path)
    print(f"{f}: {'EXISTS' if exists else 'MISSING'}")

clean_patents.csv: EXISTS
clean_inventors.csv: EXISTS
clean_companies.csv: EXISTS
clean_relationships.csv: EXISTS
